# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



In [3]:
%pip -q install duckdb
import duckdb
con = duckdb.connect()

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # exact name you set in Colab Secrets
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
[f for f in files if "daily_performance" in f]

['fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet',
 'fact_content_daily_performance/month=2025-07/data_0.parquet',
 'fact_content_daily_performance/month=2025-08/data_0.parquet',
 'fact_content_daily_performance/month=2025-09/data_0.parquet',
 'fact_content_daily_performance/month=2025-10/data_0.parquet',
 'fact_content_daily_performance/month=2025-11/data_0.parquet',
 'fact_content_daily_performance/month=2025-12/data_0.parquet',
 'fact_content_daily_performance/month=2026-01/data_0.parquet',
 'fact_content_daily_performance/month=2026-02/data_0.parquet',
 'fact_content_daily_performance/month=2026-03/data_0.parquet',
 'fact_content_daily_performance/month=2

In [5]:
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()



,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,earliest,latest
0,9841378,2026-03-01,2026-03-31


In [7]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [8]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_query_90d.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [9]:
con.sql(f"""
    SELECT MIN(window_start), MAX(window_start), MIN(window_end), MAX(window_end)
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min(window_start),max(window_start),min(window_end),max(window_end)
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30


In [10]:
con.sql(f"""
    SELECT
      COUNT(*) AS n_rows,
      SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS n_available_true,
      SUM(CASE WHEN gsc_data_available IS NULL THEN 1 ELSE 0 END) AS n_available_null,
      SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
      SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
      SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position,
      MIN(gsc_avg_position) AS min_pos, MAX(gsc_avg_position) AS max_pos,
      MIN(gsc_impressions) AS min_impr, MAX(gsc_impressions) AS max_impr,
      SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impressions,
      SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_zero_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_available_true,n_available_null,null_impressions,null_clicks,null_position,min_pos,max_pos,min_impr,max_impr,n_zero_impressions,n_zero_position
0,9841378,3611061.0,0.0,0.0,0.0,6230317.0,0.0,498.0,0,40084,6230317.0,163189.0


In [11]:
con.sql(f"""
    SELECT gsc_data_available,
           COUNT(*) AS n_rows,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS n_zero_impr,
           SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS n_null_pos
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY 1
""").df()

,gsc_data_available,n_rows,n_zero_impr,n_null_pos
0,False,6230317,6230317.0,6230317.0
1,True,3611061,0.0,0.0


In [12]:
con.sql(f"""
    SELECT gsc_data_available, gsc_impressions, gsc_clicks, COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_avg_position = 0
    GROUP BY 1, 2, 3
    ORDER BY n_rows DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,gsc_impressions,gsc_clicks,n_rows
0,True,1,0,83931
1,True,2,0,31014
2,True,3,0,15637
3,True,4,0,8995
4,True,5,0,5791
5,True,6,0,3849
6,True,7,0,2670
7,True,8,0,1930
8,True,9,0,1419
9,True,10,0,1114


In [13]:
con.sql(f"""
    SELECT
      CASE WHEN gsc_clicks = 0 THEN 'zero_clicks' ELSE 'has_clicks' END AS click_bucket,
      COUNT(*) AS n_rows,
      SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS n_position_zero
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available = TRUE
    GROUP BY 1
""").df()

,click_bucket,n_rows,n_position_zero
0,zero_clicks,3193080,162038.0
1,has_clicks,417981,1151.0


## 2. Field classification
label ingredients, context, exclude, features


## 3. Verify every claim with queries (grain, counts, missingness, windows)

## 4. Data limits



## Self-check

Before you submit, confirm each line honestly:

- [DONE] Every section above is filled — markdown thinking AND the code that backs it
- [DONE] The notebook runs top to bottom with no errors (Runtime → Run all)
- [DONE] No client names, URLs, or private queries anywhere
- [DONE] My claims use careful words: observed, measured, directional, decision-support
- [DONE] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.